In [1]:
#| default_exp htmx_compat

# htmx compatibility checker

Checker for finding obvious htmx v2/v4 compatibility issues in FastHTML source code.

In [2]:
#| export
import ast
from dataclasses import dataclass
from pathlib import Path
from fastcore.script import call_parse

HTMX_V4ONLY_ATTRS = 'action method config ignore'
HTMX_V2ONLY_ATTRS = 'disabled_elt ext history history_elt inherit params prompt request'

HTMX_V4ONLY_EVTS = 'after:init before:cleanup before:history:update before:init before:process before:history:restore after:history:push after:history:replace error'
HTMX_V2ONLY_EVTS = 'afterOnLoad afterProcessNode beforeCleanupElement beforeHistorySave beforeOnLoad beforeProcessNode beforeTransition historyCacheMiss historyRestore load oobAfterSwap oobBeforeSwap pushedIntoHistory replacedInHistory sendError swapError targetError timeout'

def _hx_attr_kwargs(xs): return {f'hx_{x}' for x in xs.split()}
def _hx_evt_kwarg(x): return 'hx_on__' + _camel2snake(x).replace(':', '_')
def _hx_evt_kwargs(xs): return {_hx_evt_kwarg(x) for x in xs.split()}

def _camel2snake(s):
    res = []
    for i,c in enumerate(s):
        if c.isupper() and i and s[i-1] not in ':_': res.append('_')
        res.append(c.lower())
    return ''.join(res)

V4ONLY_KWARGS = _hx_attr_kwargs(HTMX_V4ONLY_ATTRS) | _hx_evt_kwargs(HTMX_V4ONLY_EVTS)
V2ONLY_KWARGS = _hx_attr_kwargs(HTMX_V2ONLY_ATTRS) | _hx_evt_kwargs(HTMX_V2ONLY_EVTS)

In [3]:
V4ONLY_KWARGS

{'hx_action',
 'hx_config',
 'hx_ignore',
 'hx_method',
 'hx_on__after_history_push',
 'hx_on__after_history_replace',
 'hx_on__after_init',
 'hx_on__before_cleanup',
 'hx_on__before_history_restore',
 'hx_on__before_history_update',
 'hx_on__before_init',
 'hx_on__before_process',
 'hx_on__error'}

In [4]:
V2ONLY_KWARGS

{'hx_disabled_elt',
 'hx_ext',
 'hx_history',
 'hx_history_elt',
 'hx_inherit',
 'hx_on__after_on_load',
 'hx_on__after_process_node',
 'hx_on__before_cleanup_element',
 'hx_on__before_history_save',
 'hx_on__before_on_load',
 'hx_on__before_process_node',
 'hx_on__before_transition',
 'hx_on__history_cache_miss',
 'hx_on__history_restore',
 'hx_on__load',
 'hx_on__oob_after_swap',
 'hx_on__oob_before_swap',
 'hx_on__pushed_into_history',
 'hx_on__replaced_in_history',
 'hx_on__send_error',
 'hx_on__swap_error',
 'hx_on__target_error',
 'hx_on__timeout',
 'hx_params',
 'hx_prompt',
 'hx_request'}

In [5]:
from fastcore.test import test_eq

test_eq(_hx_attr_kwargs('action method'), {'hx_action', 'hx_method'})
test_eq(_hx_evt_kwarg('after:init'), 'hx_on__after_init')
test_eq(_hx_evt_kwarg('afterOnLoad'), 'hx_on__after_on_load')
assert 'hx_vals' not in V2ONLY_KWARGS
assert 'hx_vals' not in V4ONLY_KWARGS

In [6]:
#| export
@dataclass
class HtmxCompatIssue:
    lineno: int
    col_offset: int
    name: str
    target: int
    message: str

def _const_bool(node):
    return node.value if isinstance(node, ast.Constant) and isinstance(node.value, bool) else None

def _call_name(node):
    if isinstance(node.func, ast.Name): return node.func.id
    if isinstance(node.func, ast.Attribute): return node.func.attr
    return None

def _call_htmx4_kw(node):
    for kw in node.keywords:
        if kw.arg == 'htmx4': return _const_bool(kw.value)
    return False

In [7]:
#| export
class HtmxCompatVisitor(ast.NodeVisitor):
    def __init__(self, target=None):
        self.target = target
        self.inferred_versions = []
        self.issues = []

    def visit_Call(self, node):
        name = _call_name(node)
        if name in {'fast_app', 'FastHTML'}:
            htmx4 = _call_htmx4_kw(node)
            if htmx4 is not None: self.inferred_versions.append(4 if htmx4 else 2)
        self._check_keywords(node)
        self.generic_visit(node)

    def _resolved_target(self):
        if self.target in (2,4): return self.target
        versions = set(self.inferred_versions)
        return versions.pop() if len(versions) == 1 else None

    def _check_keywords(self, node):
        target = self._resolved_target()
        if target is None: return
        bad = V2ONLY_KWARGS if target == 4 else V4ONLY_KWARGS
        bad_version = 2 if target == 4 else 4
        for kw in node.keywords:
            if kw.arg is None: continue
            if kw.arg in bad:
                self.issues.append(HtmxCompatIssue(
                    kw.value.lineno,
                    kw.value.col_offset,
                    kw.arg,
                    target,
                    f'{kw.arg} is htmx v{bad_version}-only, but this source targets htmx v{target}.'))

In [8]:
#| export
def infer_htmx_version(src):
    "Infer htmx target version from direct fast_app/FastHTML calls, returning 2, 4, or None."
    tree = ast.parse(src)
    visitor = HtmxCompatVisitor()
    visitor.visit(tree)
    versions = set(visitor.inferred_versions)
    return versions.pop() if len(versions) == 1 else None

def check_htmx_compat(src, target=None):
    "Return static compatibility issues for FastHTML htmx kwargs in `src`."
    tree = ast.parse(src)
    visitor = HtmxCompatVisitor(target=target)
    visitor.visit(tree)
    return visitor.issues

In [9]:
#| export
def _py_files(path):
    path = Path(path)
    if path.is_file(): return [path]
    return sorted(p for p in path.rglob('*.py') if '.venv' not in p.parts and '__pycache__' not in p.parts)

def check_htmx_compat_path(path, target=None):
    "Return `(path, issue)` pairs for Python files under `path`."
    res = []
    for p in _py_files(path):
        try: src = p.read_text()
        except UnicodeDecodeError: continue
        for issue in check_htmx_compat(src, target=target): res.append((p, issue))
    return res

@call_parse
def htmx_compat_check(
    path:str='.', # File or directory to scan
    target:int=0  # Target htmx version. Use 0 to infer from fast_app/FastHTML calls.
):
    "Check FastHTML source for obvious htmx v2/v4 compatibility issues."
    issues = check_htmx_compat_path(path, target=target or None)
    for p,issue in issues:
        print(f'{p}:{issue.lineno}:{issue.col_offset + 1}: {issue.message}')
    if issues: raise SystemExit(1)

## Version inference

In [10]:
test_eq(infer_htmx_version('app, rt = fast_app(htmx4=True)'), 4)
test_eq(infer_htmx_version('app = FastHTML(htmx4=True)'), 4)
test_eq(infer_htmx_version('app, rt = fast_app()'), 2)
test_eq(infer_htmx_version('app = FastHTML(htmx4=False)'), 2)
test_eq(infer_htmx_version('fast_app(); FastHTML(htmx4=True)'), None)
test_eq(infer_htmx_version('fast_app(htmx4=USE_V4)'), None)

## v4-only attrs in a v2 app

In [11]:
src = '''
app, rt = fast_app()
Button('Go', hx_action='/items', hx_method='get')
Div(hx_config='timeout:1s')
Div(hx_ignore=True)
'''
issues = check_htmx_compat(src)
test_eq([o.name for o in issues], ['hx_action', 'hx_method', 'hx_config', 'hx_ignore'])
test_eq({o.target for o in issues}, {2})

In [12]:
issues

[HtmxCompatIssue(lineno=3, col_offset=23, name='hx_action', target=2, message='hx_action is htmx v4-only, but this source targets htmx v2.'),
 HtmxCompatIssue(lineno=3, col_offset=43, name='hx_method', target=2, message='hx_method is htmx v4-only, but this source targets htmx v2.'),
 HtmxCompatIssue(lineno=4, col_offset=14, name='hx_config', target=2, message='hx_config is htmx v4-only, but this source targets htmx v2.'),
 HtmxCompatIssue(lineno=5, col_offset=14, name='hx_ignore', target=2, message='hx_ignore is htmx v4-only, but this source targets htmx v2.')]

## v2-only attrs in a v4 app

In [13]:
src = '''
app, rt = fast_app(htmx4=True)
Div(hx_disabled_elt='this')
Div(hx_ext='ws')
Div(hx_history='false')
Div(hx_history_elt='#x')
Div(hx_inherit='*')
Div(hx_params='none')
Div(hx_prompt='Name?')
Div(hx_request='timeout:1s')
'''
issues = check_htmx_compat(src)
test_eq([o.name for o in issues], ['hx_disabled_elt', 'hx_ext', 'hx_history', 'hx_history_elt', 'hx_inherit', 'hx_params', 'hx_prompt', 'hx_request'])
test_eq({o.target for o in issues}, {4})

In [14]:
issues

[HtmxCompatIssue(lineno=3, col_offset=20, name='hx_disabled_elt', target=4, message='hx_disabled_elt is htmx v2-only, but this source targets htmx v4.'),
 HtmxCompatIssue(lineno=4, col_offset=11, name='hx_ext', target=4, message='hx_ext is htmx v2-only, but this source targets htmx v4.'),
 HtmxCompatIssue(lineno=5, col_offset=15, name='hx_history', target=4, message='hx_history is htmx v2-only, but this source targets htmx v4.'),
 HtmxCompatIssue(lineno=6, col_offset=19, name='hx_history_elt', target=4, message='hx_history_elt is htmx v2-only, but this source targets htmx v4.'),
 HtmxCompatIssue(lineno=7, col_offset=15, name='hx_inherit', target=4, message='hx_inherit is htmx v2-only, but this source targets htmx v4.'),
 HtmxCompatIssue(lineno=8, col_offset=14, name='hx_params', target=4, message='hx_params is htmx v2-only, but this source targets htmx v4.'),
 HtmxCompatIssue(lineno=9, col_offset=14, name='hx_prompt', target=4, message='hx_prompt is htmx v2-only, but this source target

## v4-only events in a v2 app

In [15]:
src = '''
app, rt = fast_app()
Div(hx_on__after_init='console.log(event)')
Div(hx_on__before_history_update='console.log(event)')
Div(hx_on__error='console.log(event)')
'''
issues = check_htmx_compat(src)
test_eq([o.name for o in issues], ['hx_on__after_init', 'hx_on__before_history_update', 'hx_on__error'])

In [16]:
issues

[HtmxCompatIssue(lineno=3, col_offset=22, name='hx_on__after_init', target=2, message='hx_on__after_init is htmx v4-only, but this source targets htmx v2.'),
 HtmxCompatIssue(lineno=4, col_offset=33, name='hx_on__before_history_update', target=2, message='hx_on__before_history_update is htmx v4-only, but this source targets htmx v2.'),
 HtmxCompatIssue(lineno=5, col_offset=17, name='hx_on__error', target=2, message='hx_on__error is htmx v4-only, but this source targets htmx v2.')]

## v2-only events in a v4 app

In [17]:
src = '''
app, rt = fast_app(htmx4=True)
Div(hx_on__after_on_load='console.log(event)')
Div(hx_on__before_cleanup_element='console.log(event)')
Div(hx_on__history_cache_miss='console.log(event)')
Div(hx_on__send_error='console.log(event)')
'''
issues = check_htmx_compat(src)
test_eq([o.name for o in issues], ['hx_on__after_on_load', 'hx_on__before_cleanup_element', 'hx_on__history_cache_miss', 'hx_on__send_error'])

In [18]:
issues

[HtmxCompatIssue(lineno=3, col_offset=25, name='hx_on__after_on_load', target=4, message='hx_on__after_on_load is htmx v2-only, but this source targets htmx v4.'),
 HtmxCompatIssue(lineno=4, col_offset=34, name='hx_on__before_cleanup_element', target=4, message='hx_on__before_cleanup_element is htmx v2-only, but this source targets htmx v4.'),
 HtmxCompatIssue(lineno=5, col_offset=30, name='hx_on__history_cache_miss', target=4, message='hx_on__history_cache_miss is htmx v2-only, but this source targets htmx v4.'),
 HtmxCompatIssue(lineno=6, col_offset=22, name='hx_on__send_error', target=4, message='hx_on__send_error is htmx v2-only, but this source targets htmx v4.')]

## Export 

In [21]:
from nbdev.export import nb_export

# Export this notebook as one standalone script next to the notebook.
# nbdev names the option `solo_nb`; this is the current equivalent of a solo export.
nb_export('compat_checker.ipynb', lib_path='.', name='htmx_compat_checker', solo_nb=True)
